# Create NCBR Awards from RAD-on

Creates Narodowe Centrum Badań i Rozwoju (NCBR, Poland's National Centre for Research and Development) awards from RAD-on, Poland's national research-information system (POLON registry). ~4-5K NCBR-financed projects.

**Prerequisites:**
- Run `scripts/local/ncbr_to_s3.py` to harvest and upload the data first.

**Data source:** https://radon.nauka.gov.pl/opendata/polon/projects (RAD-on open API, no auth).
The tracker's original source (dane.gov.pl dataset 2785) holds only 215 contracts from 2021 and was superseded by RAD-on — see the script header.

**S3 location:** `s3a://openalex-ingest/awards/ncbr/ncbr_projects.parquet`

**NCBR funder:**
- funder_id: 4320335039
- display_name: "Narodowe Centrum Badań i Rozwoju"
- ROR: https://ror.org/05pwfyy15
- DOI: 10.13039/501100005632

**Mapping notes:**
- `funder_award_id` = the native NCBR project/contract number (e.g. `LIDER/13/0049/L-9/17/NCBR/2018`, `POIR.04.01.04-00-0002/19-00`).
- `amount` = NCBR's own share (`receivedFunds` on NCBR's financingInstitutions entry), **PLN**, zeros treated as not-published.
- PI = RAD-on projectManagers (prefers kind `KP`, kierownik projektu); RAD-on delivers firstName/lastName natively.
- Coverage caveat: POLON registers projects reported by research institutions, so NCBR grants to company-only consortia are not covered.


## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.ncbr_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/ncbr/ncbr_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.ncbr_raw;

In [ ]:
%sql
-- Step 1.5: inspect raw data before transforming
DESCRIBE openalex.awards.ncbr_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.ncbr_raw LIMIT 5;

In [ ]:
%sql
-- Step 1.6 funder existence check (Path A: F4320* must return exactly 1 row)
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4320335039;

## Step 2: Create NCBR Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.ncbr_awards
USING delta
AS
WITH
ncbr_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320335039
),

awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.project_number))))) % 9000000000 as id,
        COALESCE(NULLIF(TRIM(g.title_en), ''), NULLIF(TRIM(g.title_pl), '')) as display_name,
        COALESCE(NULLIF(TRIM(g.abstract_en), ''), NULLIF(TRIM(g.abstract_pl), '')) as description,
        f.funder_id,
        TRIM(g.project_number) as funder_award_id,
        NULLIF(TRY_CAST(g.amount_pln AS DOUBLE), 0) as amount,
        'PLN' as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,
        'research' as funding_type,
        NULLIF(TRIM(g.scheme), '') as funder_scheme,
        'ncbr' as provenance,
        TRY_TO_DATE(g.start_date, 'yyyy-MM-dd') as start_date,
        TRY_TO_DATE(g.end_date, 'yyyy-MM-dd') as end_date,
        YEAR(TRY_TO_DATE(g.start_date, 'yyyy-MM-dd')) as start_year,
        YEAR(TRY_TO_DATE(g.end_date, 'yyyy-MM-dd')) as end_year,
        CASE
            WHEN g.manager_last_name IS NOT NULL AND TRIM(g.manager_last_name) != '' THEN
                struct(
                    NULLIF(TRIM(g.manager_first_name), '') as given_name,
                    TRIM(g.manager_last_name) as family_name,
                    CAST(NULL AS STRING) as orcid,
                    CAST(NULL AS DATE) as role_start,
                    struct(
                        COALESCE(NULLIF(TRIM(g.manager_institution), ''), NULLIF(TRIM(g.leader_institution), '')) as name,
                        'Poland' as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation
                )
            ELSE NULL
        END as lead_investigator,
        CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
        CAST(NULL AS STRING) as landing_page_url,
        CAST(NULL AS STRING) as doi,
        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.project_number))))) % 9000000000) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM openalex.awards.ncbr_raw g
    CROSS JOIN ncbr_funder f
    WHERE g.project_number IS NOT NULL AND TRIM(g.project_number) != ''
)
SELECT * FROM awards_transformed;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'ncbr' AND priority = 431;

-- Insert into openalex_awards_raw with priority
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    431 as priority  -- NCBR priority
FROM openalex.awards.ncbr_awards;

## Verification

In [ ]:
%sql
SELECT COUNT(*) as total_ncbr_awards FROM openalex.awards.ncbr_awards;

In [ ]:
%sql
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_abstract,
    COUNT(amount) as has_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) as pct_amount,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator) as has_pi,
    ROUND(COUNT(lead_investigator) * 100.0 / COUNT(*), 1) as pct_pi,
    MIN(amount) as min_amount,
    ROUND(AVG(amount), 0) as avg_amount,
    MAX(amount) as max_amount,
    ROUND(SUM(amount)/1e9, 2) as total_amount_billions_pln
FROM openalex.awards.ncbr_awards;

In [ ]:
%sql
-- 6.4a PI frequency check
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.ncbr_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.ncbr_awards
WHERE start_year IS NOT NULL
GROUP BY start_year ORDER BY start_year DESC LIMIT 25;